# CLL: the model inputs

The two inputs of the CloneVI fit of CLL patient 1 (CD19+ cells; Fig. 4), built from the outputs of the R steps of
`preprocess/README.md`:

| output | made from |
|---|---|
| `PT1_pos.h5ad`: fragment counts of the 5,610 kept cells in the 107,497 peaks observed in at least 10 of them (`X` and `raw.X`), the LSI clusters of Signac and the clone of the source study | `PT1_pos_pp.h5ad` of `signac_pp.R`, `PT1.tsv` of `export_mgatk.R` |
| `PT1_pos_mg.h5ad`: mtDNA coverage, mutant counts and VAF of every position in the same cells | the mgatk counts exported by `export_mgatk.R`, the cells kept by `get_filtering_num.R`, the projection and the clones of the source study |

The steps are those of the study's notebooks (the mgatk conversion, the merge of the clone definition, and the use of
the raw counts and the name `lsi_clusters` before the fit), gathered here; the files are written to `OUT` and compared
with the deposited ones (`data/ATAC/`). Run from `preprocess/CLL/`.

In [1]:
import os

import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.sparse as sp
import anndata as ad

In [2]:
MT = '../../mtscATAC-seq'                                              # raw data and the outputs of the R steps
REPRO = f'{MT}/mtscATACpaper_reproducibility/figure_CLL'                # reproducibility code of the source study
DEP = '../../data/ATAC'                                                # deposited model inputs
OUT = '../../data/ATAC/rebuilt'
os.makedirs(OUT, exist_ok=True)

## 1. Cells and clones

`get_filtering_num.R` kept the cells called by cellranger-atac with at least 1,000 fragments, at least 50% of them in
the fixed-width peaks of the source study, and a mitochondrial coverage of at least 20; `index` is the position of the
cell among the 6,000 cells of the library:

In [3]:
kept = pd.read_csv('row_del/CLL_PT1_CD19pos_del.tsv', sep='\t')
pd.Series({'cells kept': len(kept), 'fragments >= 1,000': (kept['depth'] >= 1000).all(),
           'FRiP >= 0.5': (kept['FRIP'] >= 0.5).all(), 'mtDNA coverage >= 20': (kept['mtDNAcoverage'] >= 20).all()})

cells kept              5610
fragments >= 1,000      True
FRiP >= 0.5             True
mtDNA coverage >= 20    True
dtype: object

The clones are those defined by the source study for patient 1 (`PT1_clone_definition.rds`, exported to `PT1.tsv`):

In [4]:
PT1 = pd.read_csv(f'{MT}/rds_output/clone_definition/PT1.tsv', sep='\t')
PT1.rename(columns={'cell_id': 'barcode'}, inplace=True)
PT1['cluster_id'].value_counts().sort_index()

cluster_id
1     3378
2      750
3      711
4      320
5      131
6      116
7       87
8       68
9       35
10      28
Name: count, dtype: int64

## 2. Cell-state input

`signac_pp.R` kept the peaks observed in at least 10 cells and the cells with at least 200 detected peaks, and ran
TF-IDF, LSI, UMAP and the clustering on the top 25% of peaks by count (`FindTopFeatures`, q75). Its h5ad holds the
counts of all kept peaks in `raw.X`, and the percentile of each peak's count in `var['percentile']`:

In [5]:
PT1_pos = ad.read_h5ad(f'{MT}/h5_files/PT1_pos_pp.h5ad')
PT1_pos

.../site-packages/anndata/compat/__init__.py:229: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(


AnnData object with n_obs × n_vars = 5610 × 107497
    obs: 'orig.ident', 'nCount_peaks', 'nFeature_peaks', 'peaks_snn_res.0.8', 'seurat_clusters', 'nCount_RNA', 'nFeature_RNA'
    var: 'count', 'percentile'
    uns: 'neighbors'
    obsm: 'X_lsi', 'X_umap'
    varm: 'LSI'
    obsp: 'distances'

In [6]:
pd.Series({'peaks': PT1_pos.n_vars, 'top 25% (percentile >= 0.75)': int((PT1_pos.var['percentile'] >= 0.75).sum()),
           'peaks with LSI loadings': int(np.isfinite(PT1_pos.varm['LSI']).all(axis=1).sum())})

peaks                           107497
top 25% (percentile >= 0.75)     26884
peaks with LSI loadings          26884
dtype: int64

The clone of each cell is added, the raw counts are used as `X` (the model input is the counts of all 107,497 peaks),
and the Seurat clusters are renamed `lsi_clusters`, as in the file used for the fit:

In [7]:
obs = PT1_pos.obs.reset_index(drop=False).rename(columns={'index': 'barcode'})
obs = pd.merge(obs, PT1, on='barcode', how='left')
obs.index = obs.index.astype(str)
PT1_pos.obs = obs
if '_index' in PT1_pos.raw.var.columns:                  # the peak names that SeuratDisk keeps as a column
    PT1_pos.raw.var.rename(columns={'_index': 'features'}, inplace=True)

PT1_pos.X = PT1_pos.raw.X
PT1_pos.obs['cluster_id'] = PT1_pos.obs['cluster_id'].astype('category')
PT1_pos.obs['seurat_clusters'] = PT1_pos.obs['seurat_clusters'].astype('category')
PT1_pos.obs.rename(columns={'seurat_clusters': 'lsi_clusters'}, inplace=True)
PT1_pos.var_names_make_unique()
PT1_pos.write_h5ad(f'{OUT}/PT1_pos.h5ad')
PT1_pos

AnnData object with n_obs × n_vars = 5610 × 107497
    obs: 'barcode', 'orig.ident', 'nCount_peaks', 'nFeature_peaks', 'peaks_snn_res.0.8', 'lsi_clusters', 'nCount_RNA', 'nFeature_RNA', 'cluster_id'
    var: 'count', 'percentile'
    uns: 'neighbors'
    obsm: 'X_lsi', 'X_umap'
    varm: 'LSI'
    obsp: 'distances'

Compared with the deposited input. The model reads the counts, the peaks, the cells and their clones; the LSI
clusters color Fig. 4a, and the LSI and UMAP of Signac are not used by the analyses. The SVD of the LSI fixes each
component only up to its sign, so the LSI is compared after matching the signs; the UMAP of Signac, computed from it,
differs. (`raw.var` of the deposited file lists the peaks of patient 2, written there by an error of the study's
notebook; its counts are those of patient 1.)

In [8]:
dep = ad.read_h5ad(f'{DEP}/PT1_pos.h5ad')
same = lambda a, b: (a != b).nnz == 0
lsi_sign = np.sign((dep.obsm['X_lsi'] * PT1_pos.obsm['X_lsi']).sum(axis=0))
pd.Series({'cells': np.array_equal(dep.obs['barcode'], PT1_pos.obs['barcode']),
           'peaks': np.array_equal(dep.var_names, PT1_pos.var_names),
           'counts (X)': same(dep.X, PT1_pos.X), 'counts (raw.X)': same(dep.raw.X, PT1_pos.raw.X),
           'clone (cluster_id)': np.array_equal(dep.obs['cluster_id'].astype(int), PT1_pos.obs['cluster_id'].astype(int)),
           'peak counts and features per cell': dep.obs[['nCount_peaks', 'nFeature_peaks']].equals(PT1_pos.obs[['nCount_peaks', 'nFeature_peaks']]),
           'gene activity per cell': np.allclose(dep.obs[['nCount_RNA', 'nFeature_RNA']], PT1_pos.obs[['nCount_RNA', 'nFeature_RNA']]),
           'peak percentile': np.allclose(dep.var['percentile'], PT1_pos.var['percentile']),
           'LSI clusters': np.array_equal(dep.obs['lsi_clusters'].astype(int), PT1_pos.obs['lsi_clusters'].astype(int)),
           'LSI (signs of the components matched)': np.allclose(dep.obsm['X_lsi'], PT1_pos.obsm['X_lsi'] * lsi_sign, atol=1e-6),
           'components of opposite sign': int((lsi_sign < 0).sum()),
           'UMAP of Signac': np.allclose(dep.obsm['X_umap'], PT1_pos.obsm['X_umap'], atol=1e-6)})

cells                                     True
peaks                                     True
counts (X)                                True
counts (raw.X)                            True
clone (cluster_id)                        True
peak counts and features per cell         True
gene activity per cell                    True
peak percentile                           True
LSI clusters                              True
LSI (signs of the components matched)     True
components of opposite sign                 15
UMAP of Signac                           False
dtype: object

In [9]:
pd.crosstab(dep.obs['lsi_clusters'].astype(int).values, PT1_pos.obs['lsi_clusters'].astype(int).values,
            rownames=['deposited'], colnames=['rebuilt'])

rebuilt,0,1,2,3,4,5,6
deposited,,,,,,,
0,2404,0,0,0,0,0,0
1,0,1055,0,0,0,0,0
2,0,0,721,0,0,0,0
3,0,0,0,659,0,0,0
4,0,0,0,0,571,0,0
5,0,0,0,0,0,127,0
6,0,0,0,0,0,0,73


In [10]:
del PT1_pos, dep

## 3. mtDNA input

The mgatk allele counts of the kept cells, with the projection of the source study (`umap1`, `umap2`, `cluster`) and
the clones. For each position the mutant count is the coverage minus the reads of the reference allele, and the VAF
is the mutant count divided by the coverage (0 where there is no coverage). The AnnData of the study stored the
coverage as float32 (the default of anndata before 0.9).

In [11]:
datadir = f'{MT}/rds_output/PT1_pos_mg'
matname = ['A_counts_fw', 'A_counts_rev', 'C_counts_fw', 'C_counts_rev', 'G_counts_fw', 'G_counts_rev', 'T_counts_fw', 'T_counts_rev', 'coverage']
del_index = np.array(kept['index'])

genes = pd.read_csv(os.path.join(datadir, 'genes.tsv'), sep='\t')
genes = genes.drop(columns=['strand', 'width'])
genes.index = genes.index.astype(str)
barcode = pd.read_csv(os.path.join(datadir, 'barcodes.tsv'), sep='\t', names=['barcode'])
barcode = barcode.iloc[del_index].reset_index(drop=True)

celltype = pd.read_csv(f'{REPRO}/output/projection_dfs/CLL_PT1_CD19pos_projection.tsv', sep='\t')
obs = pd.merge(barcode, celltype, how='inner', on='barcode')
obs.index = obs.index.astype(str)
assert np.array_equal(obs['barcode'], barcode['barcode'])   # every kept cell is in the projection, in the same order

mtx = sio.mmread(os.path.join(datadir, 'coverage.mtx')).toarray().T[del_index, :]
adata_mg = ad.AnnData(X=mtx.astype(np.float32), obs=obs, var=genes)
for y in range(len(matname)):
    adata_mg.layers[matname[y]] = sio.mmread(os.path.join(datadir, matname[y] + '.mtx')).toarray().T[del_index, :]
adata_mg.var_names_make_unique()
adata_mg

AnnData object with n_obs × n_vars = 5610 × 16569
    obs: 'barcode', 'umap1', 'umap2', 'cluster'
    var: 'seqnames', 'start', 'end', 'refAllele'
    layers: 'A_counts_fw', 'A_counts_rev', 'C_counts_fw', 'C_counts_rev', 'G_counts_fw', 'G_counts_rev', 'T_counts_fw', 'T_counts_rev', 'coverage'

In [12]:
# positions whose reference allele is not A, C, G or T
not_A_index = np.array(adata_mg.var.reset_index().query('refAllele!="A"').index)
not_C_index = np.array(adata_mg.var.reset_index().query('refAllele!="C"').index)
not_G_index = np.array(adata_mg.var.reset_index().query('refAllele!="G"').index)
not_T_index = np.array(adata_mg.var.reset_index().query('refAllele!="T"').index)

all_mut = 0
for base, not_index in [('A', not_A_index), ('C', not_C_index), ('G', not_G_index), ('T', not_T_index)]:
    variant = adata_mg.X - adata_mg.layers[f'{base}_counts_fw'] - adata_mg.layers[f'{base}_counts_rev']
    variant[:, not_index] = 0
    all_mut = all_mut + variant
with np.errstate(divide='ignore', invalid='ignore'):
    VAF = np.nan_to_num(all_mut / adata_mg.X)

for y in matname:
    del adata_mg.layers[y]
adata_mg.layers['all_mut'] = sp.csr_matrix(all_mut)
adata_mg.layers['VAF'] = sp.csr_matrix(VAF)
del all_mut, VAF, variant

obs = pd.merge(adata_mg.obs, PT1, on='barcode', how='inner')
obs.index = obs.index.astype(str)
adata_mg.obs = obs
adata_mg.write_h5ad(f'{OUT}/PT1_pos_mg.h5ad')
adata_mg

AnnData object with n_obs × n_vars = 5610 × 16569
    obs: 'barcode', 'umap1', 'umap2', 'cluster', 'cluster_id'
    var: 'seqnames', 'start', 'end', 'refAllele'
    layers: 'all_mut', 'VAF'

Compared with the deposited input:

In [13]:
dep = ad.read_h5ad(f'{DEP}/PT1_pos_mg.h5ad')
pd.Series({'cells': np.array_equal(dep.obs['barcode'], adata_mg.obs['barcode']),
           'positions and reference alleles': dep.var.astype(str).equals(adata_mg.var.astype(str)),
           'obs': dep.obs.astype(str).equals(adata_mg.obs.astype(str)),
           'coverage': np.array_equal(dep.X, adata_mg.X),
           'mutant counts': (dep.layers['all_mut'] != adata_mg.layers['all_mut']).nnz == 0,
           'VAF': (dep.layers['VAF'] != adata_mg.layers['VAF']).nnz == 0})

.../site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


cells                              True
positions and reference alleles    True
obs                                True
coverage                           True
mutant counts                      True
VAF                                True
dtype: bool

## 4. Numbers of the Methods

In [14]:
x = ad.read_h5ad(f'{OUT}/PT1_pos.h5ad', backed='r')
pd.DataFrame([('cells', 5610, x.n_obs), ('peaks of the model input', 107497, x.n_vars),
              ('top 25% of peaks (LSI)', 26884, int((x.var['percentile'] >= 0.75).sum())),
              ('clones of the source study', 10, x.obs['cluster_id'].nunique())],
             columns=['item', 'Methods', 'computed']).assign(match=lambda d: d['Methods'] == d['computed'])

,item,Methods,computed,match
0,cells,5610,5610,True
1,peaks of the model input,107497,107497,True
2,top 25% of peaks (LSI),26884,26884,True
3,clones of the source study,10,10,True
